# 01. Bronze: завантаження курсів валют НБУ

Цей ноутбук відповідає за перший етап конвеєра. Він отримує актуальні курси валют з API Національного банку України та зберігає відповідь у сирому вигляді в BigQuery.

На цьому етапі дані не очищуються і не змінюються. Кожен об’єкт із відповіді API зберігається як окремий JSON-текст. Це дозволяє побачити, які саме дані надійшли від джерела під час кожного запуску.

Дані записуються в режимі `WRITE_APPEND`. Тому повторний запуск не видаляє попередні рядки, а додає нову порцію даних.

## Послідовність роботи

1. Підключитися до BigQuery.
2. Створити необхідні датасети й таблицю.
3. Отримати дані з API НБУ.
4. Перетворити відповідь API на рядки DataFrame.
5. Дописати рядки в bronze-таблицю.
6. Перевірити результат завантаження.

## 1. Налаштування та підключення

Спочатку імпортуємо необхідні бібліотеки, вказуємо ідентифікатор Google Cloud проєкту та створюємо клієнт BigQuery.

Клієнт потрібен Python-коду для надсилання команд у BigQuery. Авторизація виконується через JSON-ключ сервісного акаунта, який зберігається у змінній середовища `GCP_SA_KEY`. Сам ключ не записується в ноутбук і не потрапляє до Git.

In [1]:
%pip install google-cloud-bigquery pandas pyarrow db-dtypes requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
PROJECT_ID = "nbu-bigquery-etl"   # ← ваш проєкт
LOCATION   = "EU"

DS_RAW  = "nbu_raw"       # bronze
DS_DWH  = "nbu_dwh"       # gold

In [3]:
import os, sys, json, hashlib

import requests
import pandas as pd
from google.cloud import bigquery
from datetime import datetime

pd.set_option("display.max_columns", 40)
RUN_TS   = pd.Timestamp.now(tz="UTC").floor("s")

In [4]:
creds = None

if "google.colab" in sys.modules:                  # Colab
    from google.colab import auth
    auth.authenticate_user()

elif os.environ.get("GCP_SA_KEY"):                 # ключ сервісного акаунта у змінній середовища
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: nbu-bigquery-etl


## Завдання 2. Ноутбук 01: шар bronze

### Завдання 2.1. Створення датасетів

Створюємо два датасети в регіоні `EU`. Датасет `nbu_raw` зберігатиме сирі відповіді API, а `nbu_dwh` використовуватиметься наступними ноутбуками для вимірів, снепшоту та історичної таблиці фактів.

Датасети створюються з параметром `exists_ok=True`, тому повторний запуск ноутбука не завершиться помилкою, якщо вони вже існують.

In [5]:
for ds in (DS_RAW, DS_DWH):
    d = bigquery.Dataset(f"{PROJECT_ID}.{ds}")
    d.location = LOCATION
    client.create_dataset(d, exists_ok=True)
    print("датасет:", ds)

датасет: nbu_raw
датасет: nbu_dwh


### Завдання 2.2. Створення таблиці `raw_rates`

Таблиця `nbu_raw.raw_rates` зберігатиме відповіді API НБУ в сирому вигляді. Один об’єкт із JSON-масиву відповідатиме одному рядку таблиці.

У таблиці окремо зберігаються дата курсу та момент завантаження. Це дозволяє відрізнити дату, за яку діє курс, від часу, коли дані потрапили до сховища.

Таблиця партиціюється за `business_date`, тому запити за конкретну дату можуть читати лише потрібну частину даних.

In [6]:
RAW_TABLE = f"{PROJECT_ID}.{DS_RAW}.raw_rates"

schema = [
    bigquery.SchemaField("ingested_at",   "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("business_date", "DATE"),
    bigquery.SchemaField("request_url",   "STRING"),
    bigquery.SchemaField("payload",       "STRING"),   # JSON-текст
    bigquery.SchemaField("payload_hash",  "STRING"),
]

table = bigquery.Table(RAW_TABLE, schema=schema)

table.time_partitioning = bigquery.TimePartitioning(field="business_date")

client.create_table(table, exists_ok=True)
print("таблиця готова:", RAW_TABLE)

таблиця готова: nbu-bigquery-etl.nbu_raw.raw_rates


### Завдання 2.3. Отримання курсів з API НБУ

Надсилаємо HTTP-запит до відкритого API Національного банку України та отримуємо офіційні курси валют на поточну дату.

Після отримання відповіді перевіряємо статус HTTP-запиту, перетворюємо JSON-відповідь на список Python і виводимо кількість валют та перший елемент.

In [7]:
URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"

HEADERS = {"User-Agent": "datalab-etl/1.0"}

resp = requests.get(URL, headers=HEADERS, timeout=30)
resp.raise_for_status()
data = resp.json()          # список словників

print(len(data), "валют")
print(data[0])

45 валют
{'r030': 12, 'txt': 'Алжирський динар', 'rate': 0.33613, 'cc': 'DZD', 'exchangedate': '24.08.2026', 'special': None}


### Завдання 2.4. Формування рядків bronze-таблиці

Перетворюємо кожен об’єкт із відповіді API на окремий рядок майбутньої таблиці. Повний об’єкт зберігаємо як JSON-текст у колонці `payload`, а для порівняння вмісту обчислюємо його MD5-хеш.

Дата `business_date` береться з поля `exchangedate`, тому вона показує дату, за яку діє курс. Момент `ingested_at` показує, коли дані були завантажені нашим конвеєром.

In [8]:
def to_rows(items):
    """Перетворює елементи відповіді на рядки bronze-таблиці"""
    rows = []
    for item in items:
        payload = json.dumps(item, ensure_ascii=False, sort_keys=True)
        rows.append({
            "ingested_at":   RUN_TS,
            "business_date": datetime.strptime(item['exchangedate'], "%d.%m.%Y").date(),
            "request_url":   URL,
            "payload":       payload,
            "payload_hash":  hashlib.md5(payload.encode("utf-8")).hexdigest(),
        })
    return rows

In [9]:
raw_rows = to_rows(data)

raw_df = pd.DataFrame(raw_rows)
print(len(raw_df), "рядків до запису")

45 рядків до запису


In [10]:
print(raw_df.columns.tolist())
print(raw_df.dtypes)
print(type(raw_df.loc[0, "business_date"]))

['ingested_at', 'business_date', 'request_url', 'payload', 'payload_hash']
ingested_at      datetime64[us, UTC]
business_date                 object
request_url                      str
payload                          str
payload_hash                     str
dtype: object
<class 'datetime.date'>


In [11]:
raw_df.head(5)

,ingested_at,business_date,request_url,payload,payload_hash
0,2026-08-24 08:15:17+00:00,2026-08-24,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""DZD"", ""exchangedate"": ""24.08.2026"", ""r...",59ad08be031d312443ca7bf7865f938d
1,2026-08-24 08:15:17+00:00,2026-08-24,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""AUD"", ""exchangedate"": ""24.08.2026"", ""r...",303d0342d3da995395ffee3e0abfbb64
2,2026-08-24 08:15:17+00:00,2026-08-24,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""BDT"", ""exchangedate"": ""24.08.2026"", ""r...",2f9558fb3e4c48c6e6eb28cc2cc9b857
3,2026-08-24 08:15:17+00:00,2026-08-24,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""CAD"", ""exchangedate"": ""24.08.2026"", ""r...",342c80ce9fcece8604a5c58263045715
4,2026-08-24 08:15:17+00:00,2026-08-24,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""CNY"", ""exchangedate"": ""24.08.2026"", ""r...",b5aaa5dcd62ac10b0e3613acc256723c


### Завдання 2.5. Запис у BigQuery

Завантажуємо підготовлений DataFrame у таблицю `nbu_raw.raw_rates` у режимі `WRITE_APPEND`.

Цей режим не видаляє попередні рядки. Кожен запуск додає нову порцію даних, тому bronze зберігає історію всіх завантажень.

Завантаження виконується як окреме завдання BigQuery. Метод `.result()` очікує його завершення, тому перевірка кількості рядків запускається тільки після фактичного запису даних.

In [12]:
cfg = bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_APPEND")

client.load_table_from_dataframe(raw_df, RAW_TABLE, job_config=cfg).result()

print("у таблиці всього:", client.get_table(RAW_TABLE).num_rows, "рядків")

e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


у таблиці всього: 90 рядків


### Завдання 2.6. Перевірка завантаження

Виконуємо агрегований SQL-запит до bronze-таблиці. Для кожної `business_date` порівнюємо загальну кількість рядків із кількістю унікальних `payload_hash`.

Це дозволяє відрізнити кількість подій завантаження від кількості унікальних варіантів отриманих даних.

In [13]:
sql = f"""
SELECT business_date,
       COUNT(*) AS rows_cnt,
       COUNT(DISTINCT payload_hash) AS uniq_cnt
FROM `{RAW_TABLE}`
GROUP BY business_date
ORDER BY business_date
"""
check_df = client.query(sql).to_dataframe()
print(len(check_df), "рядків у підсумковому результаті")
check_df.head(3)

1 рядків у підсумковому результаті


e:\Data Engineering\DataLab\Python\dataeng\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,business_date,rows_cnt,uniq_cnt
0,2026-08-24,90,45


### Завдання 2.7. Перевірка повторного запуску

Я запустив ноутбук удруге. Після повторного завантаження кількість рядків за `business_date` збільшилася з 45 до 90, тому що режим `WRITE_APPEND` додав нову порцію до вже наявних даних.

Кількість унікальних `payload_hash` залишилася 45. API повернув ті самі дані, тому однакові JSON-об’єкти отримали однакові хеші.

Для bronze така поведінка є нормальною. Таблиця зберігає кожну подію завантаження, а видалення повторів відбудеться на наступному етапі конвеєра.

In [14]:
print("01_bronze завершено успішно")

01_bronze завершено успішно
